In [14]:
import pandas as pd
import numpy as np
import os
import math
import seaborn as sns
from sklearn.linear_model import LinearRegression
import matplotlib.pyplot as plt
from scipy import stats
from statsmodels.stats.power import GofChisquarePower

In [15]:
humans = pd.read_csv('./data/phenotype/all.panel',delim_whitespace=True)


for file_name in os.listdir('./data/genotype')[0:2]:
    for ratio in list(range(40)):
        ratio = ratio/2
        
        snp_num = 0
        true_mean = 0
        true_effect = 5
        true_sd = true_effect*ratio
        iteration = 200
        min_power = 0.9
        
        geno = pd.read_pickle(f"./data/genotype/{file_name}")
        geno[geno.columns[snp_num]].value_counts()
        num_maj = geno[geno.columns[snp_num]].value_counts()[1]
        num_het = geno[geno.columns[snp_num]].value_counts()[0]
        try:
            num_min = geno[geno.columns[snp_num]].value_counts()[-1]
        except Exception as e:
            num_min = 0
        total_humans = num_maj + num_het + num_min
        maf = (num_min*2 + num_het)/(total_humans*2)
        """
        Loop for nul hypothesis
        """
        estimated_effect_nul = []
        estimated_intercept_nul = []
        SE_nul = []
        p_nul = []
        
        i = 0
        for i in range(iteration):
            mean_min = true_mean
            mean_het = true_mean
            mean_maj = true_mean
            sigma_min = true_sd
            sigma_het = true_sd
            sigma_maj = true_sd
            pheno = []
            for var in list(geno[geno.columns[snp_num]]):
                if var == 1:
                    pheno.append(np.random.normal(mean_maj, sigma_maj, 1)[0])
                elif var == 0:
                    pheno.append(np.random.normal(mean_het, sigma_het, 1)[0])
                else:
                    pheno.append(np.random.normal(mean_min, sigma_min, 1)[0])
            df_pheno = pd.DataFrame(data = {"geno": list(geno[geno.columns[snp_num]]), "pheno_null": pheno})

            X = df_pheno[['geno']]
            y = df_pheno[['pheno_null']]
            reg = LinearRegression().fit(X, y)
            estimated_effect_nul.append(reg.coef_[0][0])
            estimated_intercept_nul.append(reg.intercept_[0])

            # Calculating p values and standard error
            N = len(X)
            p = len(X.columns) + 1
            X_with_intercept = np.empty(shape=(N, p), dtype=float)
            X_with_intercept[:, 0] = 1
            X_with_intercept[:, 1:p] = X.values

            params = np.append(reg.intercept_,reg.coef_)
            predictions = reg.predict(X)
            new_y = (y-predictions)**2
            newX = pd.DataFrame({"Constant":np.ones(len(X))}).join(pd.DataFrame(list(X['geno'])))
            MSE = (sum(list(new_y['pheno_null'])))/(len(newX)-len(newX.columns))

            var_b = MSE*(np.linalg.inv(np.dot(newX.T,newX)).diagonal())


            sd_b = np.sqrt(var_b)
            ts_b = params/ sd_b

            p_values =[2*(1-stats.t.cdf(np.abs(i),(len(newX)-len(newX.columns)))) for i in ts_b]

            SE_nul.append(sd_b[1])
            p_nul.append(p_values[1])

19.5


In [19]:
ratio

19.5